# F-003-3: YOLO-Fastest v1.1 学習・量子化 Notebook

本Notebookは、転倒検出用 YOLO-Fastest v1.1 モデルを Google Colab 上で学習し、
TFLite INT8 形式にエクスポートするための手順を提供する。

## 前提条件
- Google Colab (GPU ランタイム: T4 推奨)
- Google Drive にデータセットをアップロード済み

## ワークフロー
1. GPU確認・環境構築
2. データセットのアップロード
3. Darknetビルド・設定ファイル作成
4. アンカー計算
5. モデル学習
6. 精度評価 (mAP)
7. ONNX変換
8. TFLite FP32/INT8変換
9. 成果物ダウンロード

---
## Step 0: GPU確認

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('\n=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')
    print('CPU でも学習可能ですが、非常に時間がかかります。')

---
## Step 1: 環境構築

AlexeyAB/darknet をビルドし、変換ツールをインストールする。

In [ ]:
import os
WORK_DIR = '/content/yolo_fastest'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'作業ディレクトリ: {WORK_DIR}')

In [ ]:
%%bash
# Darknet ビルド (AlexeyAB fork)
cd /content/yolo_fastest
if [ ! -d darknet ]; then
    git clone https://github.com/AlexeyAB/darknet.git
    cd darknet
    # GPU + CUDNN + OpenCV 有効化
    sed -i 's/GPU=0/GPU=1/' Makefile
    sed -i 's/CUDNN=0/CUDNN=1/' Makefile
    sed -i 's/OPENCV=0/OPENCV=1/' Makefile
    sed -i 's/LIBSO=0/LIBSO=1/' Makefile
    make -j$(nproc) 2>&1 | tail -5
    echo '=== Darknet ビルド完了 ==='
else
    echo 'Darknet は既にビルド済みです'
fi

In [ ]:
# Python ライブラリのインストール
!pip install -q onnx onnxruntime onnx-tf tensorflow-cpu Pillow tqdm

---
## Step 2: データセット準備

### 方法A: Google Driveからアップロード (推奨)

ローカルPCで以下を実行してZIPを作成し、Google Driveにアップロードしてください:

```bash
cd mimamori-sense/dataset/merged
zip -r ~/fall_detection_dataset.zip images/ labels/ train.txt valid.txt test.txt obj.data obj.names
```

作成したZIPファイルをGoogle Driveのルートにアップロードしてください。

### 方法B: スクリプトで再ダウンロード

Roboflow APIキーが必要です。方法Aを推奨します。

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content/yolo_fastest

# ===== 設定 =====
# Google Drive上のZIPファイルパス
DATASET_ZIP="/content/drive/MyDrive/fall_detection_dataset.zip"
# ================

if [ ! -d dataset ]; then
    if [ -f "$DATASET_ZIP" ]; then
        echo "データセットZIPを展開中..."
        mkdir -p dataset
        unzip -q "$DATASET_ZIP" -d dataset/
        echo "=== 展開完了 ==="
    else
        echo "ERROR: $DATASET_ZIP が見つかりません"
        echo "Google DriveにZIPファイルをアップロードしてください"
        exit 1
    fi
fi

# データセット確認
echo ""
echo "=== データセット構成 ==="
for split in train val test; do
    img_count=$(ls dataset/images/$split/ 2>/dev/null | wc -l)
    lbl_count=$(ls dataset/labels/$split/ 2>/dev/null | wc -l)
    echo "  $split: images=$img_count, labels=$lbl_count"
done

In [ ]:
%%bash
cd /content/yolo_fastest

# Darknet用のパスリストを生成 (絶対パスに変換)
DATASET_DIR="/content/yolo_fastest/dataset"

for split in train val test; do
    img_dir="$DATASET_DIR/images/$split"
    out_file="$DATASET_DIR/${split}.txt"
    find "$img_dir" -type f \( -name '*.jpg' -o -name '*.png' -o -name '*.jpeg' \) | sort > "$out_file"
    echo "$split: $(wc -l < $out_file) entries -> $out_file"
done

---
## Step 3: 設定ファイル作成

YOLO-Fastest v1.1 の cfg ファイルを 192x192x1 Grayscale, 1クラス (person) 用にカスタマイズする。

In [ ]:
%%writefile /content/yolo_fastest/dataset/obj.names
person

In [ ]:
# obj.data 生成
obj_data = """classes = 1
train = /content/yolo_fastest/dataset/train.txt
valid = /content/yolo_fastest/dataset/val.txt
names = /content/yolo_fastest/dataset/obj.names
backup = /content/yolo_fastest/backup/
"""
os.makedirs('/content/yolo_fastest/backup', exist_ok=True)
with open('/content/yolo_fastest/dataset/obj.data', 'w') as f:
    f.write(obj_data)
print('obj.data 作成完了')

In [ ]:
%%writefile /content/yolo_fastest/yolo-fastest-person.cfg
[net]
batch=64
subdivisions=16
width=192
height=192
channels=1
momentum=0.949
decay=0.0005
angle=0
saturation=0
exposure=1.5
hue=0

learning_rate=0.01
burn_in=300
max_batches=20000
policy=steps
steps=16000,18000
scales=.1,.1

# ========== Backbone ==========
[convolutional]
batch_normalize=1
filters=8
size=3
stride=2
pad=1
activation=leaky

# DW Conv
[convolutional]
batch_normalize=1
filters=8
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=8
size=3
stride=1
pad=1
groups=8
activation=leaky

[convolutional]
batch_normalize=1
filters=4
size=1
stride=1
pad=1
activation=leaky

# Shuffle Block 1 - stride 2
[convolutional]
batch_normalize=1
filters=24
size=1
stride=1
pad=1
activation=leaky

[convolutional]
batch_normalize=1
filters=24
size=3
stride=2
pad=1
groups=24
activation=leaky

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-4

[convolutional]
batch_normalize=1
filters=4
size=3
stride=2
pad=1
groups=4
activation=leaky

[convolutional]
batch_normalize=1
filters=16
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

# channel shuffle (simulated)
[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

# Shuffle Block 2 - stride 2
[convolutional]
batch_normalize=1
filters=32
size=3
stride=2
pad=1
groups=32
activation=leaky

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-4

[maxpool]
size=3
stride=2
padding=1

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=64
size=1
stride=1
pad=1
activation=leaky

# Shuffle Block 3
[convolutional]
batch_normalize=1
filters=64
size=3
stride=1
pad=1
groups=64
activation=leaky

[convolutional]
batch_normalize=1
filters=64
size=1
stride=1
pad=1
activation=leaky

# Shuffle Block 4 - stride 2
[convolutional]
batch_normalize=1
filters=64
size=3
stride=2
pad=1
groups=64
activation=leaky

[convolutional]
batch_normalize=1
filters=64
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-4

[maxpool]
size=3
stride=2
padding=1

[convolutional]
batch_normalize=1
filters=64
size=1
stride=1
pad=1
activation=leaky

[route]
layers=-1,-4

[convolutional]
batch_normalize=1
filters=128
size=1
stride=1
pad=1
activation=leaky

# Shuffle Block 5
[convolutional]
batch_normalize=1
filters=128
size=3
stride=1
pad=1
groups=128
activation=leaky

[convolutional]
batch_normalize=1
filters=128
size=1
stride=1
pad=1
activation=leaky

# ========== Detection Head ==========

# Branch 0 (6x6 grid - large objects)
[convolutional]
batch_normalize=1
filters=64
size=1
stride=1
pad=1
activation=leaky

[convolutional]
size=1
stride=1
pad=1
filters=18
activation=linear

[yolo]
mask=3,4,5
anchors=14,26, 19,37, 28,55, 38,77, 47,97, 61,126
classes=1
num=6
jitter=.3
ignore_thresh=.5
truth_thresh=1
random=0
scale_x_y=1.05

# Upsample for Branch 1
[route]
layers=-4

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

[upsample]
stride=2

# Concat with earlier feature map
[route]
layers=-1,-16

[convolutional]
batch_normalize=1
filters=32
size=1
stride=1
pad=1
activation=leaky

# Branch 1 (12x12 grid - small objects)
[convolutional]
size=1
stride=1
pad=1
filters=18
activation=linear

[yolo]
mask=0,1,2
anchors=14,26, 19,37, 28,55, 38,77, 47,97, 61,126
classes=1
num=6
jitter=.3
ignore_thresh=.5
truth_thresh=1
random=0
scale_x_y=1.05

---
## Step 4: アンカー計算

personデータセットに最適なアンカーボックスを計算する。

**注意:** 計算結果に応じて cfg ファイルのアンカー値を更新する必要がある。

In [ ]:
%%bash
cd /content/yolo_fastest/darknet
./darknet detector calc_anchors \
    /content/yolo_fastest/dataset/obj.data \
    -num_of_clusters 6 \
    -width 192 -height 192 \
    -show 2>&1 || echo '(アンカー計算完了 - 上記の anchors を cfg に反映してください)'

In [ ]:
# === アンカー更新 ===
# 上のセルで出力されたアンカー値をここに入力してください。
# 例: "10,20, 15,35, 25,50, 35,75, 50,100, 65,130"
# 空文字の場合はデフォルト値（顔認識ベースライン）を使用します。

NEW_ANCHORS = ""  # <-- ここにアンカー値を貼り付け

if NEW_ANCHORS.strip():
    cfg_path = '/content/yolo_fastest/yolo-fastest-person.cfg'
    with open(cfg_path) as f:
        content = f.read()
    old_anchors = '14,26, 19,37, 28,55, 38,77, 47,97, 61,126'
    content = content.replace(old_anchors, NEW_ANCHORS.strip())
    with open(cfg_path, 'w') as f:
        f.write(content)
    print(f'アンカーを更新しました: {NEW_ANCHORS.strip()}')
else:
    print('デフォルトアンカーを使用します: 14,26, 19,37, 28,55, 38,77, 47,97, 61,126')

---
## Step 5: モデル学習

学習を実行する。T4 GPU で約2-4時間を想定。

学習中に Colab のセッションが切れないよう注意すること。

**ヒント:**
- チャートは `chart_yolo-fastest-person.png` に出力される
- ベストモデルは `backup/yolo-fastest-person_best.weights` に保存される
- 途中で中断した場合、最後のチェックポイントから再開可能

In [ ]:
%%bash
cd /content/yolo_fastest/darknet

# 学習実行
./darknet detector train \
    /content/yolo_fastest/dataset/obj.data \
    /content/yolo_fastest/yolo-fastest-person.cfg \
    -map -dont_show \
    2>&1 | tail -20

echo ""
echo "=== 学習完了 ==="
ls -la /content/yolo_fastest/backup/

In [ ]:
# 学習曲線の表示
import matplotlib.pyplot as plt
from IPython.display import Image, display
import os

chart = '/content/yolo_fastest/darknet/chart_yolo-fastest-person.png'
if os.path.exists(chart):
    display(Image(filename=chart))
else:
    print('学習チャートが見つかりません')

### (オプション) 学習の中断・再開

学習が中断された場合、最後のチェックポイントから再開できる。

In [ ]:
# # === 中断後の再開 (必要な場合のみ実行) ===
# !cd /content/yolo_fastest/darknet && \
#     ./darknet detector train \
#     /content/yolo_fastest/dataset/obj.data \
#     /content/yolo_fastest/yolo-fastest-person.cfg \
#     /content/yolo_fastest/backup/yolo-fastest-person_last.weights \
#     -map -dont_show

---
## Step 6: 精度評価 (mAP)

In [ ]:
%%bash
cd /content/yolo_fastest/darknet

WEIGHTS="/content/yolo_fastest/backup/yolo-fastest-person_best.weights"

if [ -f "$WEIGHTS" ]; then
    echo "=== Validation セットでの mAP 評価 ==="
    ./darknet detector map \
        /content/yolo_fastest/dataset/obj.data \
        /content/yolo_fastest/yolo-fastest-person.cfg \
        "$WEIGHTS" \
        2>&1 | grep -E '(mean_average|class_id|precision|recall|F1)'
else
    echo "ERROR: $WEIGHTS が見つかりません。学習を先に実行してください。"
fi

---
## Step 7: ONNX変換

Darknet weights を PyTorch 経由で ONNX に変換する。

In [ ]:
%%bash
cd /content/yolo_fastest
if [ ! -d pytorch-YOLOv4 ]; then
    git clone https://github.com/Tianxiaomo/pytorch-YOLOv4.git
fi
echo 'pytorch-YOLOv4 (darknet2onnx変換ツール) 準備完了'

In [ ]:
import sys
sys.path.insert(0, '/content/yolo_fastest/pytorch-YOLOv4')

import os
CFG = '/content/yolo_fastest/yolo-fastest-person.cfg'
WEIGHTS = '/content/yolo_fastest/backup/yolo-fastest-person_best.weights'
ONNX_OUT = '/content/yolo_fastest/yolo-fastest-person.onnx'

if os.path.exists(WEIGHTS):
    # darknet2onnx 変換
    from demo_darknet2onnx import main as darknet2onnx
    try:
        darknet2onnx(CFG, WEIGHTS, ONNX_OUT, 1)  # batch_size=1
        print(f'\n=== ONNX エクスポート完了: {ONNX_OUT} ===')
        print(f'サイズ: {os.path.getsize(ONNX_OUT)/1024:.1f} KB')
    except Exception as e:
        print(f'darknet2onnx 変換でエラー: {e}')
        print('代替手法で変換を試みます...')
        # 代替: demo_darknet2onnx.py をコマンドラインで実行
        os.system(f'cd /content/yolo_fastest/pytorch-YOLOv4 && '
                  f'python demo_darknet2onnx.py {CFG} {WEIGHTS} {ONNX_OUT} 1')
else:
    print(f'ERROR: {WEIGHTS} が見つかりません')

---
## Step 8: TFLite FP32/INT8 変換

In [ ]:
import numpy as np
import os, glob
from PIL import Image

ONNX_PATH = '/content/yolo_fastest/yolo-fastest-person.onnx'
FP32_PATH = '/content/yolo_fastest/yolo-fastest-person_fp32.tflite'
INT8_PATH = '/content/yolo_fastest/yolo-fastest-person_int8.tflite'
CAL_DIR = '/content/yolo_fastest/dataset/images/val'
IMG_SIZE = 192
CAL_COUNT = 200  # キャリブレーション画像数

# --- Step 8a: ONNX → TFLite FP32 ---
if os.path.exists(ONNX_PATH):
    try:
        import onnx
        from onnx_tf.backend import prepare
        import tensorflow as tf

        print('=== Step 8a: ONNX → TFLite FP32 ===')
        saved_model_dir = '/content/yolo_fastest/saved_model'
        model = onnx.load(ONNX_PATH)
        tf_rep = prepare(model)
        tf_rep.export_graph(saved_model_dir)

        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        tflite_model = converter.convert()
        with open(FP32_PATH, 'wb') as f:
            f.write(tflite_model)
        print(f'FP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')

        # --- Step 8b: INT8 量子化 ---
        print('\n=== Step 8b: INT8 量子化 ===')
        cal_images = sorted(glob.glob(os.path.join(CAL_DIR, '*.jpg')))[:CAL_COUNT]
        if not cal_images:
            cal_images = sorted(glob.glob(os.path.join(CAL_DIR, '*.png')))[:CAL_COUNT]
        print(f'キャリブレーション画像: {len(cal_images)}枚')

        def representative_dataset():
            for img_path in cal_images:
                img = Image.open(img_path).convert('L').resize((IMG_SIZE, IMG_SIZE))
                arr = np.array(img, dtype=np.float32).reshape(1, IMG_SIZE, IMG_SIZE, 1) / 255.0
                yield [arr]

        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        int8_kb = os.path.getsize(INT8_PATH) / 1024
        print(f'INT8 TFLite: {int8_kb:.1f} KB')
        print(f'アリーナ制約 (432KB) に収まる: {int8_kb <= 432}')

    except Exception as e:
        print(f'ERROR: {e}')
else:
    print(f'ERROR: {ONNX_PATH} が見つかりません')

In [ ]:
# INT8モデルの検査
import tensorflow as tf
import numpy as np

INT8_PATH = '/content/yolo_fastest/yolo-fastest-person_int8.tflite'

if os.path.exists(INT8_PATH):
    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    print('=== INT8 モデル検査 ===')
    print(f'ファイルサイズ: {os.path.getsize(INT8_PATH)/1024:.1f} KB')

    for label, details in [('Input', interp.get_input_details()),
                           ('Output', interp.get_output_details())]:
        print(f'\n--- {label} ---')
        for i, d in enumerate(details):
            print(f'  [{i}] {d["name"]} shape={d["shape"]} dtype={d["dtype"]}')
            qp = d.get('quantization_parameters', {})
            sc = qp.get('scales', np.array([]))
            zp = qp.get('zero_points', np.array([]))
            if len(sc) > 0:
                print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')
else:
    print(f'{INT8_PATH} が見つかりません')

---
## Step 9: 成果物ダウンロード

学習済みモデルを Google Drive にコピーし、ローカルにダウンロードする。

In [ ]:
%%bash
# Google Drive に成果物をコピー
OUTPUT_DIR="/content/drive/MyDrive/fall_detection_model"
mkdir -p "$OUTPUT_DIR"

# 各ファイルをコピー (存在する場合のみ)
for f in \
    /content/yolo_fastest/backup/yolo-fastest-person_best.weights \
    /content/yolo_fastest/backup/yolo-fastest-person_final.weights \
    /content/yolo_fastest/yolo-fastest-person.cfg \
    /content/yolo_fastest/yolo-fastest-person.onnx \
    /content/yolo_fastest/yolo-fastest-person_fp32.tflite \
    /content/yolo_fastest/yolo-fastest-person_int8.tflite \
    /content/yolo_fastest/darknet/chart_yolo-fastest-person.png; do
    if [ -f "$f" ]; then
        cp "$f" "$OUTPUT_DIR/"
        echo "Copied: $(basename $f) ($(du -h $f | cut -f1))"
    fi
done

echo ""
echo "=== Google Drive に保存完了 ==="
echo "場所: $OUTPUT_DIR"
ls -lh "$OUTPUT_DIR/"

In [ ]:
# INT8 TFLite モデルを直接ダウンロード
from google.colab import files
import os

INT8_PATH = '/content/yolo_fastest/yolo-fastest-person_int8.tflite'
if os.path.exists(INT8_PATH):
    files.download(INT8_PATH)
    print('INT8モデルのダウンロードを開始しました')
else:
    print('INT8モデルが見つかりません。Step 8 を先に実行してください。')

---
## まとめ

### 生成される成果物

| ファイル | 説明 |
|---|---|
| `yolo-fastest-person_best.weights` | 学習済みDarknet重み (最良mAP) |
| `yolo-fastest-person.cfg` | ネットワーク設定ファイル |
| `yolo-fastest-person.onnx` | ONNX形式モデル |
| `yolo-fastest-person_fp32.tflite` | TFLite FP32モデル |
| `yolo-fastest-person_int8.tflite` | TFLite INT8量子化モデル |
| `chart_yolo-fastest-person.png` | 学習曲線チャート |

### 次のステップ

1. INT8モデルのサイズが432KB以内であることを確認
2. RUHMI `mcu_quantize.py` または `mcu_deploy.py` でEthos-U55向けC99コードを生成 (Step 5)
3. 生成コードを `e2studio_CPU0/src/ai_application/mera/` に配置
4. 実機 (EK-RA8P1) での動作確認